# Julia notebook to compute the symbolic solution to the frictional geostrophic equations with a specified relaxation buoyancy field and surface wind stress.
twnh 

Dec '25   Write `ExampleSolution_v0.5.ipynb` <br>
Jul '26.  Write `FrictionalGeostrophicEquationSolution.ipynb`

This notebook solves the non-dimensional, linear, time-independent frictional geostrophic equations:

\begin{align}
- f  v & = - \frac{\partial p}{\partial x} + Ek \,  \frac{d}{dz} \left( \nu \frac{d u}{d z} \right), \\
  f  u & = - \frac{\partial p}{\partial y} + Ek \,  \frac{d}{dz} \left( \nu \frac{d v}{d z} \right) , 
\end{align}
for $u(z), v(z)$, given viscosity $\nu = \nu (z)$ (customized so $\nu = 1$ at the moment), and boundary conditions
\begin{align}
\text{Surface~}z = 0:
\begin{cases}
\displaystyle \nu \partial_z u & = \tau^x \\
\displaystyle  \nu \partial_z v & = \tau^y 
\end{cases}
 \\
\text{Bottom~}z = -H:
\begin{cases}
u & = 0  \\
v & = 0 
\end{cases} ,
\end{align}
where $(\tau^x, \tau^y)$ is the applied surface stress.
The pressure $p$ is due to the surface pressure field $p_s(x,y)$ and buoyancy field $b(x,y,z)$:

\begin{align}
p(x,y,z) & = p_s(x,y) + \int_{z}^{0} b(x,y,z') \; dz' , \\
\implies p_b(x, y) & \equiv p(x, y, z=-H(x,y)) = p_s(x,y) + \int_{-H(x,y)}^{0} b(x,y,z') \; dz' .
\end{align}

This notebook also derives the equation satisfied by the surface pressure field $p_s(x,y)$.

In [ ]:
using SymPy

### Define symbols:

In [ ]:
# Geometry symbolic parameters:
z, ξ   = symbols("z ξ",   real=true, negative=true)   # Vertical coordinate and source location (both in [-H,0])
x, y   = symbols("x y",   real=true)                  # Horizontal coordinates
H      = SymFunction("H", real=true, positive=true)   # Domain depth H(x,y)
geometry_params = (H(x,y), z, ξ)

Ek     = symbols("Ek ",   real=true, positive=true)   # Ekman number

# Frictional geostrophic equations symbolic parameters:
f      = symbols("f",       real=true, positive=true) # Coriolis parameter
ϕ      = symbols("ϕ",       real=true, positive=true) # Viscosity parameter
τs     = SymFunction("τs",  complex=true)             # Complex surface stress
psg    = SymFunction("psg", complex=true)             # Surface pressure gradient (∂/∂x + i ∂/∂y) pₛ(x,y)

# Buoyancy symbolic parameters:
b      = SymFunction("b", real=true)                # Buoyancy field b(x,y,z)
ψ      = symbols("ψ",     real=true, positive=true) # Diffusivity parameters

# Define viscosity profile here:
ν0     = symbols("ν0",    real=true, positive=true) # Viscosity parameters
ν = ν0 + 0.0 * z                                    # Constant viscosity profile. Aspire to depth-varying profile in future.
ν = ν.subs(ν0, 1)                                 # Set ν0 = 1.0 for now. Setting it not equal to 1.0 is sometimes useful for testing.
uv_params = (f, Ek, ν) 

# Convenience variable:
root_iϕ = ϕ*sympy.sqrt(im) ;

### Compute pressure fields for the specified buoyancy field $b$:

In [ ]:
# Define buoyancy gradient field:
bg     = SymFunction("bg",    complex=true)         # Buoyancy gradient

# Compute pressure field:
pbarog = SymPy.integrate(bg(x,y,ξ),(ξ,z,0))         # Baroclinic presure gradient 
pg     = psg(x,y) + pbarog                          # Total pressure gradient
pbotg  = pg.subs(z,-H(x,y))                         # Bottom pressure gradient

### Function to solve the frictional geostrophic equation using a Green's function: This is where `SymPy` is unavoidable as there's no `Symbolics` equivalent to `dsolve` as of July 2026.

In [ ]:
function compute_Guv(uv_params, geometry_params)
    # Setup symbols and parameters:
    f, Ek, ν = uv_params
    H, z, ξ = geometry_params
    uv      = SymFunction("uv")
    A       = symbols("A",real=true)                                                  # Unknown coefficient in the Green's function solution

    # 0. Define the ODE for d/dz(uv(z)) = uv(z):
    ode = Eq(-im * f * uv(z) + Ek * diff(ν * diff(uv(z),z),z), 0)
    
    # 1. Solve for $G_- (z)$ on $-H \le z \le ξ$:
    Gₘ = dsolve(ode, uv(z), ics = Dict(uv(z).subs(z,-H(x,y))=>0)).rhs
    @assert sympy.simplify(ode.lhs.subs(uv(z),Gₘ)) == 0                               # Check solution
    # Replace constant names because otherwise they can interfere with the constants from the next dsolve below.
    const_names = collect([string(s) for s in Gₘ.free_symbols if occursin(r"^C\d+", string(s))])
    Gₘ = Gₘ.subs(const_names[1],A)
    @assert sympy.simplify(Gₘ.subs(z,-H(x,y))) == 0                                   # Check bottom BC

    # 2. Solve for $G_+(z)$ on  $ξ \le z \le 0$:
    Gₚ = dsolve(ode, uv(z), ics = Dict(diff(uv(z),z).subs(z,0)=>0)).rhs
    @assert sympy.simplify(ode.lhs.subs(uv(z),Gₚ)) == 0                               # Check solution
    @assert sympy.simplify(diff(Gₚ,z).subs(z,0)) == 0                                 # Check surface BC

    # 3. Compute Wronskian $W(z)$:
    W = Gₘ * diff(Gₚ, z) - Gₚ * diff(Gₘ, z)

    # 4. Compute Green's function $G(z; \xi)$:
    Gm = Gₘ * Gₚ.subs(z,ξ) / (ν.subs(z,ξ) * W.subs(z,ξ))
    Gp = Gₘ.subs(z,ξ) * Gₚ / (ν.subs(z,ξ) * W.subs(z,ξ))

    # Check continuity and jump condition:
    @assert Gm.subs(z,ξ) - Gp.subs(z,ξ) == 0
    @assert sympy.simplify(diff(Gm, z).subs(z,ξ) - diff(Gp, z).subs(z,ξ)) + 1/(ν.subs(z,ξ)) == 0

    # #5. Define piecewise Green's function:
    G = sympy.Piecewise((Gm, Le(z,ξ)), (Gp, Ge(z,ξ)))
    
    # Check boundary conditions:
    @assert sympy.simplify(diff(G,z).subs(z,0).subs(ξ,-H//2)) == 0
    @assert sympy.simplify(G.subs(z,-H(x,y)).subs(ξ,-H//2)) == 0

    # Final sympy.simplify (to cancel constants). Avoid sympy.simplify in general because it's not always reproducible.
    G = sympy.simplify(G)
    return G
end ;

### Compute the velocity Green's function:

In [ ]:
Guv_sym = compute_Guv(uv_params,geometry_params) 
Guv = Guv_sym.subs(f,ϕ^2 * Ek)
println("1.1 Symbolic computation of Guv completed.")
display(Guv)

In [ ]:
# Used for stress-driven part of the velocity:
Guv0 = Guv.subs(ξ,0)
Latex_Guv0 = (im * sympy.sqrt(im) * exp(-root_iϕ * z)/(ϕ * (1 + exp(2 * root_iϕ * H(x,y))))) *
(exp(2 * root_iϕ * (z + H(x,y))) - 1)
resid = Guv0 - Latex_Guv0
@assert resid == 0
println("1.2 LaTeX Guv0 expression matches.")
display(Latex_Guv0)

# Used for surface-pressure-driven part of the velocity:
tmp = sympy.integrate(sympy.expand(Guv), (ξ, -H(x,y), 0)).args[1].args[1]
max_obj = sympy.Max(z, -H(x, y))
Guv_int_wrt_ξ = tmp.subs(max_obj, z)
Latex_Guv_int_wrt_ξ = (im / (ϕ^2 * (1 + exp(2 * root_iϕ * H(x,y)))) ) *
(1 + exp(2 * root_iϕ * H(x,y)) - exp(root_iϕ * (z + H(x,y))) - exp(root_iϕ * (H(x,y) - z)))
resid = Guv_int_wrt_ξ - Latex_Guv_int_wrt_ξ
@assert resid == 0
println("1.3 LaTeX Guv integral wrt ξ expression matches.")
display(Latex_Guv_int_wrt_ξ)

In [ ]:
# Used for bottom boundary stress computation:
tmp  = diff(Guv, z)
tmp2 = tmp.subs(z, -H(x,y))
dGuv_dz_at_bottom = tmp2.args[1].args[1]
Latex_dGuv_dz_at_bottom = (-exp(-root_iϕ * (ξ - H(x,y)))) / (1 + exp(2 * root_iϕ * H(x,y))) * (1 + exp(2 * root_iϕ * ξ))
resid = dGuv_dz_at_bottom - Latex_dGuv_dz_at_bottom
@assert resid == 0
println("1.4 LaTeX d/dz Guv(z,ξ) @ z = -H(x,y) matches.")
display(Latex_dGuv_dz_at_bottom)

Latex_dGuv_dz_at_bottom_ξ0 = Latex_dGuv_dz_at_bottom.subs(ξ,0)
resid = dGuv_dz_at_bottom.subs(ξ,0) - Latex_dGuv_dz_at_bottom_ξ0
@assert resid == 0
println("1.5 LaTeX d/dz Guv(z,0) @ z = -H(x,y) matches.")
display(Latex_dGuv_dz_at_bottom_ξ0)

In [ ]:
# Used for baroclinic-pressure-driven part of the velocity:
tmp = sympy.integrate(sympy.expand(Guv * exp(ψ * ξ)),(ξ,-H(x,y),0))
Guv_exp_psi_xi_int_wrt_ξ = tmp.subs(max_obj, z)
Latex_Guv_exp_psi_xi_int_wrt_ξ = (1/(ψ^2 - im * ϕ^2)) * (
    ( im * sympy.sqrt(im) * ψ * exp(      ψ * H(x,y)) * (exp(2 * root_iϕ * (z + H(x,y))) - 1) -
                            ϕ * exp(root_iϕ * H(x,y)) * (exp(2 * root_iϕ *  z          ) + 1)
    ) * exp(-root_iϕ * z - ψ * H(x,y))
    /(ϕ * (1 + exp(2 * root_iϕ * H(x,y))))
    + exp(ψ * z)
)
resid = sympy.expand(Guv_exp_psi_xi_int_wrt_ξ - Latex_Guv_exp_psi_xi_int_wrt_ξ)
@assert resid == 0
println("1.6 LaTeX Guv * exp(ψ * z) integral matches.")
display(Latex_Guv_exp_psi_xi_int_wrt_ξ)

### Symbolic computation of flow $u(z),v(z), \tau_b, U, V$

In [ ]:
# Compute flow field u(z), v(z):
𝔲1 = SymPy.integrate(sympy.expand(Guv * pg.subs(z,ξ)),(ξ,-H(x,y),0))
𝔲1 = 𝔲1.subs(max_obj,z)             # Buoyancy-driven part of the velocity
𝔲1 = simplify(𝔲1)
𝔲2 = - Guv0 * τs(x,y)               # Stress-driven part of the velocity
𝔲2 = simplify(𝔲2)
𝔲 = 𝔲1 + 𝔲2
println("2.1 Symbolic computation of 𝔲 completed.")
display(𝔲)

In [ ]:
# Check that the final expression for 𝔲 satisfies the original differential equation:
tmp1 = -im * ϕ^2 * 𝔲1 + diff(ν * diff(𝔲1,z),z)
tmp1 = simplify(tmp1)
tmp2 = -im * ϕ^2 * 𝔲2 + diff(ν * diff(𝔲2,z),z)
tmp2 = simplify(tmp2)
@assert tmp1 + tmp2 == pg
println("2.2 Final expression for 𝔲 satisfies the original differential equation.")

In [ ]:
# Compute bottom stress on fluid:
τb = - ν * diff(𝔲,z).subs(z,-H(x,y))
    
# Check expression for surface stress on fluid:
@assert sympy.simplify(diff(𝔲1,z).subs(z,0)) == 0     # Pressure-driven part of surface stress vanishes
tmp = ν * diff(𝔲,z).subs(z,0)
@assert sympy.simplify(tmp - τs(x,y)) == 0
println("2.3 Expression for surface stress on fluid is correct.")

In [ ]:
# Check integral of surface pressure term for bottom stress:
tmp_τ_integrand = sympy.expand( (1 + exp(2*root_iϕ*ξ)) * exp(-root_iϕ*(ξ-H(x,y))) / (1 + exp(2*root_iϕ*H(x,y))) )
tmp = sympy.simplify(sympy.integrate(tmp_τ_integrand,(ξ,-H(x,y),0)).args[1].args[1])
Latex_tmp = (1/root_iϕ) * sympy.tanh(root_iϕ*H(x,y))
@assert tmp == Latex_tmp
println("2.4 LaTeX integral of the τb surface pressure term matches.")

# Check S(x) function:
Latex_S_fn = Latex_tmp
XXX = symbols("XXX")                            # SymPy can't pull the constant psg factor outside the integral.
τb_tmp = sympy.expand(τb.subs(psg(x,y),XXX).doit())
psg_coeff = τb_tmp.coeff(XXX)
testS = sympy.simplify(Latex_S_fn - sympy.simplify(psg_coeff))
@assert testS == 0
println("2.5 LaTeX S(x) function matches.")
display(Latex_S_fn)

# Check T(x) function:
function make_Latex_T_fn(int_bg_term)
    tmp_T_integrand2 = sympy.expand(tmp_τ_integrand * int_bg_term)
    tmp_T_fn = ( sympy.integrate(tmp_T_integrand2,(ξ,-H(x,y),0)) - τs(x,y)*(2*exp(root_iϕ*H(x,y)))/(1 + exp(2*root_iϕ*H(x,y))))
    return tmp_T_fn
end
int_bg_term = sympy.integrate(bg(x,y,ξ),(ξ,ξ,0))        # Generic integral of bg.
Latex_T_fn = make_Latex_T_fn(int_bg_term)

τb_tmp2 = (τb_tmp - psg_coeff * XXX).doit()
testT = τb_tmp2 - Latex_T_fn
YYY = symbols("YYY")                            # SymPy can't work on the buried bg integral, so substitute for it
testT = testT.subs(bg(x,y,ξ),YYY)
testT = sympy.expand(testT).doit()
@assert testT == 0
println("2.6 LaTeX T(x) function matches.")
display(Latex_T_fn)

In [ ]:
# Compute depth-integrated flow:
𝔘 = SymPy.integrate(sympy.expand(𝔲),(z,-H(x,y),0))

# Check A(x) function:
Latex_A_fn = -H(x,y)/root_iϕ^2 - (1/root_iϕ^3)*((1 - exp(2*root_iϕ*H(x,y))) / (1 + exp(2*root_iϕ*H(x,y)) ) )
𝔘_tmp = sympy.expand(𝔘.subs(psg(x,y),XXX).doit())
psg_coeff = 𝔘_tmp.coeff(XXX)
testA = sympy.simplify(Latex_A_fn - psg_coeff)
@assert testA == 0
println("2.5 LaTeX A(x) function matches.")
display(Latex_A_fn)

In [ ]:
# Check integral of surface pressure term for 𝔘:
tmp_ps_integrand = sympy.expand(exp(-root_iϕ*ξ)*(exp(root_iϕ*ξ) - exp(root_iϕ*H(x,y)))*(exp(root_iϕ*(ξ + H(x,y))) - 1))
tmp = sympy.integrate(tmp_ps_integrand,(ξ,-H(x,y),0))
Latex_tmp = - H(x,y) * (exp(2*root_iϕ*H(x,y)) + 1) + (1/root_iϕ)*(exp(2*root_iϕ*H(x,y)) - 1)
@assert tmp == Latex_tmp
println("2.6 LaTeX integral of the 𝔘 surface pressure term matches.")

In [ ]:
# Check B(x) function:
function make_Latex_B_fn(int_bg_term)
    tmp_B_integrand2 = sympy.expand(tmp_ps_integrand * int_bg_term)
    tmp_B_fn = (1/(root_iϕ^2*(1+exp(2*root_iϕ*H(x,y))))) * ( sympy.integrate(tmp_B_integrand2,(ξ,-H(x,y),0)) + τs(x,y)*(exp(root_iϕ*H(x,y)) - 1)^2)
    return tmp_B_fn
end
Latex_B_fn = make_Latex_B_fn(int_bg_term)
𝔘_tmp2 = 𝔘_tmp.subs(XXX,0)      # remove the pressure-gradient term
testB = (𝔘_tmp2 - Latex_B_fn)
testB = testB.subs(bg(x,y,ξ),YYY)
testB = sympy.expand(testB).doit()
@assert testB == 0
println("2.7 LaTeX B(x) function matches.")
display(Latex_B_fn)

### Make plots

In [ ]:
using CairoMakie

# -----------------------------
# Parameters
# -----------------------------
H  = 1.0
f  = 1.0
Ek = 0.05

varϕ = sqrt(1im * f / Ek)   # complex, as in the paper

zgrid  = range(-H, 0.0, length=250)
xigrid = range(-H, 0.0, length=250)


# -----------------------------
# Green's function G_u(z; ξ)
# -----------------------------
function G_u(z, ξ; H=1.0, varϕ=1.0 + 0im)
    # Julia's principal branch for i^(3/2)
    const i32 = (1im)^(3/2)
    pref = i32 * exp(-varϕ * (z + ξ)) / (2varϕ * (1 + exp(2varϕ * H)))

    if z <= ξ
        return pref * (-exp(2varϕ * ξ) + exp(2varϕ * (z + H)) + exp(2varϕ * (z + ξ + H)) - 1)
    else
        return pref * (-exp(2varϕ * z) + exp(2varϕ * (ξ + H)) + exp(2varϕ * (z + ξ + H)) - 1)
    end
end

# -----------------------------
# Integral of G_u over ξ
# This multiplies the surface pressure gradient term.
# -----------------------------
function Gu_int(z; H=1.0, varϕ=1.0 + 0im)
    num = 1 + exp(2varϕ*H) - exp(varϕ*(z + H)) - exp(varϕ*(H - z))
    return 1im * num / (varϕ^2 * (1 + exp(2varϕ*H)))
end

Gu_re(z, ξ; H=1.0, varϕ=1.0 + 0im) = real(G_u(z, ξ; H=H, varϕ=varϕ))
Gu_im(z, ξ; H=1.0, varϕ=1.0 + 0im) = imag(G_u(z, ξ; H=H, varϕ=varϕ))

# -----------------------------
# Precompute matrices
# -----------------------------
Gmat_re = [Gu_re(z, ξ; H=H, varϕ=varϕ) for z in zgrid, ξ in xigrid]
Gmat_im = [Gu_im(z, ξ; H=H, varϕ=varϕ) for z in zgrid, ξ in xigrid]

clim_re = maximum(abs, Gmat_re)
clim_im = maximum(abs, Gmat_im)

# -----------------------------
# Figure 1: line cuts, real + imag together
# -----------------------------
ξvals = [-0.95H, -0.75H, -0.5H, -0.25H, -0.05H]

fig1 = Figure(size=(950, 650))
ax1 = Axis(fig1[1, 1],
    xlabel = "G_u(z; ξ)",
    ylabel = "depth z",
    title  = "Velocity Green's function line cuts"
)

for ξ in ξvals
    Gre = [Gu_re(z, ξ; H=H, varϕ=varϕ) for z in zgrid]
    Gim = [Gu_im(z, ξ; H=H, varϕ=varϕ) for z in zgrid]

    lines!(ax1, Gre, zgrid, linewidth=2, label="Re, ξ=$(round(ξ, digits=2))")
    lines!(ax1, Gim, zgrid, linewidth=2, linestyle=:dash, label="Im, ξ=$(round(ξ, digits=2))")
end

axislegend(ax1, position=:rb, nbanks=2)
ylims!(ax1, -H, 0.0)

# -----------------------------
# Figure 2: heatmaps side by side
# -----------------------------
fig2 = Figure(size=(1400, 650))

ax2 = Axis(fig2[1, 1],
    xlabel = "forcing depth ξ",
    ylabel = "response depth z",
    title  = "Re(G_u(z; ξ))"
)

hm_re = heatmap!(ax2, xigrid, zgrid, Gmat_re,
    colormap = :balance,
    colorrange = (-clim_re, clim_re)
)
contour!(ax2, xigrid, zgrid, Gmat_re, color=:black, linewidth=0.8)
Colorbar(fig2[1, 2], hm_re, label = "Re(G_u)")

ax3 = Axis(fig2[1, 3],
    xlabel = "forcing depth ξ",
    title  = "Im(G_u(z; ξ))"
)

hm_im = heatmap!(ax3, xigrid, zgrid, Gmat_im,
    colormap = :balance,
    colorrange = (-clim_im, clim_im)
)
contour!(ax3, xigrid, zgrid, Gmat_im, color=:black, linewidth=0.8)
Colorbar(fig2[1, 4], hm_im, label = "Im(G_u)")

linkyaxes!(ax2, ax3)
ylims!(ax2, -H, 0.0)

# -----------------------------
# Display
# -----------------------------
fig1
fig2

In [ ]:
using CairoMakie
using LaTeXStrings

# Evaluate on the depth grid
Gu_int_re_vals = [Gu_int_re(z; H=H, ϕ=ϕ) for z in zgrid]
Gu_int_im_vals = [Gu_int_im(z; H=H, ϕ=ϕ) for z in zgrid]
Gu_int_abs_vals = [Gu_int_abs(z; H=H, ϕ=ϕ) for z in zgrid]

# -----------------------------
# Figure: integral kernel multiplying surface pressure gradient
# -----------------------------
fig5 = Figure(size=(700, 650))

ax5 = Axis(fig5[1, 1],
    xlabel = "value",
    ylabel = "depth z",
    title  = L"\int_{-H}^0 G_u(z;\xi)\,d\xi"
)

lines!(ax5, Gu_int_re_vals, zgrid, linewidth=2, label="Re")
lines!(ax5, Gu_int_im_vals, zgrid, linewidth=2, linestyle=:dash, label="Im")
lines!(ax5, Gu_int_abs_vals, zgrid, linewidth=2, linestyle=:dot, label="|.|")

axislegend(ax5, position=:rb)
ylims!(ax5, -H, 0.0)

fig5

In [ ]:

# ============================================================
# Precompute matrices for G_u(z;ξ)
# ============================================================
Gmat_re = [Gu_re(z, ξ; H=H, ϕ=ϕ) for z in zgrid, ξ in xigrid]
Gmat_im = [Gu_im(z, ξ; H=H, ϕ=ϕ) for z in zgrid, ξ in xigrid]

clim_re = maximum(abs, Gmat_re)
clim_im = maximum(abs, Gmat_im)

# Colors for the cut curves
cut_colors = Makie.wong_colors()
cut_cols = cut_colors[1:length(ξvals)]

# ============================================================
# Figure 1: Full kernel G_u(z;ξ)
# ============================================================
fig1 = Figure(size=(1700, 700), fontsize=16)

# --- Left: line cuts ---
ax1 = Axis(fig1[1, 1],
    xlabel = "G_u(z; ξ)",
    ylabel = "z",
    title  = "Line cuts of G_u(z;ξ)"
)

for (k, ξ) in enumerate(ξvals)
    c = cut_cols[k]
    Gre = [Gu_re(z, ξ; H=H, ϕ=ϕ) for z in zgrid]
    Gim = [Gu_im(z, ξ; H=H, ϕ=ϕ) for z in zgrid]

    lines!(ax1, Gre, zgrid, color=c, linewidth=2.5,
           label = "Re, ξ = $(round(ξ, digits=2))")
    lines!(ax1, Gim, zgrid, color=c, linewidth=2.5, linestyle=:dash,
           label = "Im, ξ = $(round(ξ, digits=2))")
end

axislegend(ax1, position=:rb, nbanks=2)
ylims!(ax1, -H, 0.0)

# --- Real heatmap ---
ax2 = Axis(fig1[1, 2],
    xlabel = "ξ",
    ylabel = "z",
    title  = "Re(G_u(z;ξ))"
)

hm_re = heatmap!(ax2, xigrid, zgrid, Gmat_re,
    colormap = :balance,
    colorrange = (-clim_re, clim_re)
)
contour!(ax2, xigrid, zgrid, Gmat_re, color=:black, linewidth=0.7)

# show cut locations
for ξ in ξvals
    vlines!(ax2, ξ, color=:white, linestyle=:dash, linewidth=1.5)
end

Colorbar(fig1[1, 3], hm_re, label = "Re(G_u)")

# --- Imag heatmap ---
ax3 = Axis(fig1[1, 4],
    xlabel = "ξ",
    ylabel = "",
    title  = "Im(G_u(z;ξ))"
)

hm_im = heatmap!(ax3, xigrid, zgrid, Gmat_im,
    colormap = :balance,
    colorrange = (-clim_im, clim_im)
)
contour!(ax3, xigrid, zgrid, Gmat_im, color=:black, linewidth=0.7)

# show cut locations
for ξ in ξvals
    vlines!(ax3, ξ, color=:white, linestyle=:dash, linewidth=1.5)
end

Colorbar(fig1[1, 5], hm_im, label = "Im(G_u)")

linkyaxes!(ax1, ax2, ax3)
ylims!(ax2, -H, 0.0)
ylims!(ax3, -H, 0.0)

# ============================================================
# Figure 2: Integral kernel and surface kernel side by side
# ============================================================
Gu_int_re_vals = [Gu_int_re(z; H=H, ϕ=ϕ) for z in zgrid]
Gu_int_im_vals = [Gu_int_im(z; H=H, ϕ=ϕ) for z in zgrid]

Gu0_re_vals = [Gu0_re(z; H=H, ϕ=ϕ) for z in zgrid]
Gu0_im_vals = [Gu0_im(z; H=H, ϕ=varϕ) for z in zgrid]

fig2 = Figure(size=(1300, 650), fontsize=16)

# Left panel: integral kernel
ax4 = Axis(fig2[1, 1],
    xlabel = "value",
    ylabel = "z",
    title  = "∫₋ᴴ⁰ Gᵤ(z;ξ) dξ"
)

lines!(ax4, Gu_int_re_vals, zgrid, linewidth=2.5, color=:blue, label="Re")
lines!(ax4, Gu_int_im_vals, zgrid, linewidth=2.5, color=:blue, linestyle=:dash, label="Im")

axislegend(ax4, position=:rb)
ylims!(ax4, -H, 0.0)

# Right panel: surface kernel
ax5 = Axis(fig2[1, 2],
    xlabel = "value",
    ylabel = "",
    title  = "Gᵤ(z;0)"
)

lines!(ax5, Gu0_re_vals, zgrid, linewidth=2.5, color=:blue, label="Re")
lines!(ax5, Gu0_im_vals, zgrid, linewidth=2.5, color=:blue, linestyle=:dash, label="Im")

axislegend(ax5, position=:rb)
ylims!(ax5, -H, 0.0)

fig2


In [ ]:
fig1

In [ ]:
using CairoMakie
using QuadGK

CairoMakie.activate!()

# -------------------------------------------------------------------
# Parameters
# -------------------------------------------------------------------
H  = 1.0
Γ  = 1.0

Ek1 = 0.25
Ek2 = 0.025
Ek_vals = [Ek1, Ek2]
ψ_vals  = [sqrt(Γ / Ek) for Ek in Ek_vals]

zgrid = range(-H, 0.0, length=256)

# -------------------------------------------------------------------
# Publication-style theme
# -------------------------------------------------------------------
set_theme!(
    Theme(
        fontsize = 16,
        fonts = (; regular = "TeX Gyre Heros"),
        Axis = (
            titlesize = 18,
            xlabelsize = 16,
            ylabelsize = 16,
            xticklabelsize = 13,
            yticklabelsize = 13,
            xgridvisible = true,
            ygridvisible = true,
            xgridstyle = :dash,
            ygridstyle = :dash,
            xgridcolor = (:gray70, 0.35),
            ygridcolor = (:gray70, 0.35),
            spinewidth = 1.2,
            xticksize = 5,
            yticksize = 5,
            topspinevisible = true,
            rightspinevisible = true,
        ),
        Legend = (
            framevisible = false,
            labelsize = 13,
            titlesize = 14,
        ),
        Lines = (
            linewidth = 2.5,
        )
    )
)

# -------------------------------------------------------------------
# Complex horizontal velocity U(z) = u(z) + i v(z)
# Generic forcing F(ξ) can represent (∂x + i∂y)p or other complex forcing
# -------------------------------------------------------------------
function U_profile(z; H=1.0, ψ=1.0, Ek=1.0, τ=0.0 + 0.0im, Ffunc=ξ -> 0.0 + 0.0im)
    integrand(ξ) = G_u(z, ξ; H=H, ψ=ψ, Ek=Ek) * Ffunc(ξ)
    I, _ = quadgk(integrand, -H, 0.0)
    surf = G_u(z, 0.0; H=H, ψ=ψ, Ek=Ek) * τ
    return I - surf
end

# -------------------------------------------------------------------
# Example complex forcing functions
# Replace these with the actual forcing structure from your model
# -------------------------------------------------------------------
F1(ξ) = exp(4ξ / H) + im * ξ
F2(ξ) = ξ + im * exp(4ξ / H)

forcing_families = [
    ("complex forcing 1", F1),
    ("complex forcing 2", F2)
]

colors = Dict(
    "complex forcing 1" => :royalblue,
    "complex forcing 2" => :darkorange
)

linestyles = Dict(
    0.0 => :solid,
    0.1 => :dash
)

# Example surface stresses
τ_vals = [0.0 + 0.0im, 0.1 + 0.05im]

# -------------------------------------------------------------------
# Compute velocity profiles
# -------------------------------------------------------------------
function compute_velocity_profiles(Ek, ψ)
    profiles = Dict{Tuple{String, ComplexF64}, Vector{ComplexF64}}()
    for (fname, Ffunc) in forcing_families
        for τ in τ_vals
            profiles[(fname, τ)] = [
                U_profile(z; H=H, ψ=ψ, Ek=Ek, τ=τ, Ffunc=Ffunc)
                for z in zgrid
            ]
        end
    end
    return profiles
end

# -------------------------------------------------------------------
# Figure layout
# -------------------------------------------------------------------
fig = Figure(size = (1400, 1600))

Label(fig[0, 1:2],
    "Horizontal velocity profiles for two Ekman numbers",
    fontsize = 22,
    font = :bold
)

# -------------------------------------------------------------------
# Top row: Ek1
# -------------------------------------------------------------------
profiles1 = compute_velocity_profiles(Ek1, ψ_vals[1])

ax11 = Axis(fig[1, 1],
    xlabel = "u(z)",
    ylabel = "depth z/H",
    title  = "u(z), Ek = $(Ek1)",
    aspect = 1.0
)

ax12 = Axis(fig[1, 2],
    xlabel = "v(z)",
    ylabel = "",
    title  = "v(z), Ek = $(Ek1)",
    aspect = 1.0
)

for (fname, _) in forcing_families
    c = colors[fname]
    for τ in τ_vals
        vals = profiles1[(fname, τ)]
        lines!(ax11, real.(vals), zgrid, color=c, linewidth=2.6, linestyle=τ == 0.0 + 0.0im ? :solid : :dash)
        lines!(ax12, imag.(vals), zgrid, color=c, linewidth=2.6, linestyle=τ == 0.0 + 0.0im ? :solid : :dash)
    end
end

# -------------------------------------------------------------------
# Bottom row: Ek2
# -------------------------------------------------------------------
profiles2 = compute_velocity_profiles(Ek2, ψ_vals[2])

ax21 = Axis(fig[2, 1],
    xlabel = "u(z)",
    ylabel = "depth z/H",
    title  = "u(z), Ek = $(Ek2)",
    aspect = 1.0
)

ax22 = Axis(fig[2, 2],
    xlabel = "v(z)",
    ylabel = "",
    title  = "v(z), Ek = $(Ek2)",
    aspect = 1.0
)

for (fname, _) in forcing_families
    c = colors[fname]
    for τ in τ_vals
        vals = profiles2[(fname, τ)]
        lines!(ax21, real.(vals), zgrid, color=c, linewidth=2.6, linestyle=τ == 0.0 + 0.0im ? :solid : :dash)
        lines!(ax22, imag.(vals), zgrid, color=c, linewidth=2.6, linestyle=τ == 0.0 + 0.0im ? :solid : :dash)
    end
end

# -------------------------------------------------------------------
# Axis linking and cosmetics
# -------------------------------------------------------------------
linkyaxes!(ax11, ax21)
linkyaxes!(ax12, ax22)

for ax in (ax11, ax12, ax21, ax22)
    ax.topspinevisible = true
    ax.rightspinevisible = true
end

# -------------------------------------------------------------------
# Legends
# -------------------------------------------------------------------
legend_u = Legend(fig[3, 1],
    [
        LineElement(color = colors["complex forcing 1"], linewidth = 3),
        LineElement(color = colors["complex forcing 2"], linewidth = 3),
        LineElement(color = :black, linestyle = :solid, linewidth = 3),
        LineElement(color = :black, linestyle = :dash, linewidth = 3),
    ],
    [
        "forcing 1",
        "forcing 2",
        "τ = 0",
        "τ ≠ 0"
    ],
    title = "Velocity panel",
    framevisible = false,
    orientation = :horizontal,
    nbanks = 2
)

legend_v = Legend(fig[3, 2],
    [
        LineElement(color = colors["complex forcing 1"], linewidth = 3),
        LineElement(color = colors["complex forcing 2"], linewidth = 3),
        LineElement(color = :black, linestyle = :solid, linewidth = 3),
        LineElement(color = :black, linestyle = :dash, linewidth = 3),
    ],
    [
        "forcing 1",
        "forcing 2",
        "τ = 0",
        "τ ≠ 0"
    ],
    title = "Velocity panel",
    framevisible = false,
    orientation = :horizontal,
    nbanks = 2
)

# -------------------------------------------------------------------
# Spacing
# -------------------------------------------------------------------
colgap!(fig.layout, 1, 10)
rowgap!(fig.layout, 1, 4)
rowgap!(fig.layout, 2, 2)

fig